Import Libraries

In [ ]:
# Cell 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

print("="*50)
print("📊 PHASE 4: BEHAVIOR ANALYSIS")
print("="*50)
print("✅ All imports successful!")

Define Detection Functions

In [ ]:
# Cell 2: Define Detection Functions

def detect_speeding(speed, speed_limit=60):
    """
    Detect if vehicle is speeding
    """
    if speed > speed_limit * 1.1:  # 10% over limit
        return True, speed - speed_limit
    return False, 0

def detect_harsh_braking(acceleration, threshold=-3.0):
    """
    Detect harsh braking (negative acceleration)
    """
    if acceleration < threshold:  # More negative than -3 m/s²
        return True, abs(acceleration)
    return False, 0

def detect_harsh_cornering(lateral_accel, threshold=2.0):
    """
    Detect harsh cornering
    """
    if abs(lateral_accel) > threshold:  # More than 2 m/s²
        return True, abs(lateral_accel)
    return False, 0

def calculate_driver_score(speed_violations, braking_violations, cornering_violations):
    """
    Calculate driver scorecard (0-100)
    """
    # Start with 100, deduct for each violation
    score = 100
    score -= speed_violations * 5
    score -= braking_violations * 3
    score -= cornering_violations * 2
    
    # Keep score between 0 and 100
    return max(0, min(100, score))

print("✅ Detection functions defined!")
print("\n📐 Detection Rules:")
print("   - Speeding: > 10% over speed limit")
print("   - Harsh Braking: Deceleration > 3 m/s²")
print("   - Harsh Cornering: Lateral accel > 2 m/s²")
print("   - Score: 100 - deductions")

Generate Synthetic Driving Data

In [ ]:
# Cell 3: Generate Synthetic Driving Data

def generate_driving_data(duration_seconds=300, sample_rate=1):
    """
    Generate synthetic driving data with realistic patterns.
    
    Parameters:
    - duration_seconds: How long to simulate (seconds)
    - sample_rate: Data points per second
    
    Returns:
    - pandas DataFrame with driving data
    """
    np.random.seed(42)
    
    num_samples = duration_seconds * sample_rate
    timestamps = [datetime.now() + timedelta(seconds=i/sample_rate) 
                  for i in range(num_samples)]
    
    # Generate speed with patterns (city driving)
    speed = np.zeros(num_samples)
    for i in range(num_samples):
        # Simulate varying speeds
        if i < 50:  # Starting
            speed[i] = np.random.normal(20, 5)
        elif i < 100:  # Accelerating
            speed[i] = np.random.normal(40, 10)
        elif i < 150:  # Highway
            speed[i] = np.random.normal(70, 5)
        elif i < 200:  # Braking
            speed[i] = np.random.normal(30, 10)
        else:  # Mixed
            speed[i] = np.random.normal(50, 15)
    
    speed = np.clip(speed, 0, 90)  # Limit speed
    
    # Calculate acceleration (derivative of speed)
    acceleration = np.zeros(num_samples)
    for i in range(1, num_samples):
        acceleration[i] = speed[i] - speed[i-1] + np.random.normal(0, 0.5)
    
    # Random harsh events
    acceleration[50:60] = -5  # Harsh braking
    acceleration[120:130] = -4  # Another harsh braking
    acceleration[180:185] = 6   # Hard acceleration
    
    # Lateral acceleration (for cornering)
    lateral_accel = np.random.normal(0, 1, num_samples)
    lateral_accel[75:85] = 3   # Harsh cornering
    lateral_accel[160:170] = -2.5  # Another harsh cornering
    
    # Create DataFrame
    df = pd.DataFrame({
        'timestamp': timestamps,
        'speed': speed,
        'acceleration': acceleration,
        'lateral_accel': lateral_accel
    })
    
    return df

# Generate data
print("📊 Generating synthetic driving data...")
df = generate_driving_data(duration_seconds=300, sample_rate=1)
print(f"✅ Generated {len(df)} data points")
print(f"📊 Data shape: {df.shape}")
df.head(5)

 Explore the Data

In [ ]:
# Cell 4: Explore the Data

print("\n📊 Data Summary:")
print(df.describe())

# Plot speed profile
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Speed
axes[0].plot(df['timestamp'], df['speed'], linewidth=1)
axes[0].axhline(y=66, color='r', linestyle='--', label='Speed limit (60 km/h)')
axes[0].set_title('Vehicle Speed Over Time')
axes[0].set_ylabel('Speed (km/h)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Acceleration
axes[1].plot(df['timestamp'], df['acceleration'], linewidth=1)
axes[1].axhline(y=-3, color='r', linestyle='--', label='Harsh braking threshold')
axes[1].axhline(y=3, color='orange', linestyle='--', label='Hard accel threshold')
axes[1].set_title('Acceleration Over Time')
axes[1].set_ylabel('Acceleration (m/s²)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Lateral acceleration
axes[2].plot(df['timestamp'], df['lateral_accel'], linewidth=1)
axes[2].axhline(y=2, color='r', linestyle='--', label='Harsh cornering threshold')
axes[2].axhline(y=-2, color='r', linestyle='--')
axes[2].set_title('Lateral Acceleration Over Time')
axes[2].set_ylabel('Lateral Accel (m/s²)')
axes[2].set_xlabel('Time')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Detect Violations

In [ ]:
# Cell 5: Detect Violations

print("🔍 Scanning for violations...")

# Track violations
speeding_events = []
braking_events = []
cornering_events = []

# Speed limit
SPEED_LIMIT = 60  # km/h

for idx, row in df.iterrows():
    # Check speeding
    is_speeding, over_limit = detect_speeding(row['speed'], SPEED_LIMIT)
    if is_speeding:
        speeding_events.append({
            'time': row['timestamp'],
            'speed': row['speed'],
            'over_limit': over_limit
        })
    
    # Check harsh braking
    is_braking, brake_strength = detect_harsh_braking(row['acceleration'])
    if is_braking:
        braking_events.append({
            'time': row['timestamp'],
            'deceleration': brake_strength
        })
    
    # Check harsh cornering
    is_cornering, corner_strength = detect_harsh_cornering(row['lateral_accel'])
    if is_cornering:
        cornering_events.append({
            'time': row['timestamp'],
            'lateral_accel': corner_strength
        })

print(f"\n📊 Violations Detected:")
print(f"   Speeding Events:    {len(speeding_events)}")
print(f"   Harsh Braking:      {len(braking_events)}")
print(f"   Harsh Cornering:    {len(cornering_events)}")

if len(speeding_events) > 0:
    print(f"   Max Speeding:       {max(speeding_events, key=lambda x: x['speed'])['speed']:.1f} km/h")
if len(braking_events) > 0:
    print(f"   Max Braking:        {max(braking_events, key=lambda x: x['deceleration'])['deceleration']:.1f} m/s²")
if len(cornering_events) > 0:
    print(f"   Max Cornering:      {max(cornering_events, key=lambda x: x['lateral_accel'])['lateral_accel']:.1f} m/s²")

Calculate Driver Scorecard

In [ ]:
# Cell 6: Calculate Driver Scorecard

print("📊 Calculating driver scorecard...")

# Calculate score
score = calculate_driver_score(
    speed_violations=len(speeding_events),
    braking_violations=len(braking_events),
    cornering_violations=len(cornering_events)
)

print(f"\n📊 DRIVER SCORECARD")
print("="*50)
print(f"   Speeding Events:    {len(speeding_events)}")
print(f"   Harsh Braking:      {len(braking_events)}")
print(f"   Harsh Cornering:    {len(cornering_events)}")
print("   " + "-"*40)
print(f"   Total Violations:   {len(speeding_events) + len(braking_events) + len(cornering_events)}")
print(f"   Driver Score:       {score}/100")

# Rating
if score >= 90:
    rating = "⭐ EXCELLENT"
    color = 'green'
elif score >= 70:
    rating = "👍 GOOD"
    color = 'blue'
elif score >= 50:
    rating = "⚠️ NEEDS IMPROVEMENT"
    color = 'orange'
else:
    rating = "🚨 UNSAFE"
    color = 'red'

print(f"   Rating:             {rating}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Violations pie chart
violations = [len(speeding_events), len(braking_events), len(cornering_events)]
labels = ['Speeding', 'Harsh Braking', 'Harsh Cornering']
colors = ['#ff6b6b', '#ffd93d', '#6bcb77']
ax1.pie(violations, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax1.set_title('Violation Breakdown')

# Score gauge
from matplotlib.patches import Wedge
import matplotlib.patches as mpatches

# Create a simple gauge
gauge_colors = ['red', 'orange', 'yellow', 'green']
gauge = ax2.barh([0], [score], color='green', height=0.5)
ax2.set_xlim(0, 100)
ax2.set_xticks([0, 20, 40, 60, 80, 100])
ax2.set_yticks([])
ax2.set_xlabel('Score')
ax2.set_title(f'Driver Score: {score}/100')

plt.tight_layout()
plt.show()

Phase 4 Summary

In [ ]:
# Cell 7: Phase 4 Summary
print("\n" + "="*50)
print("📊 PHASE 4: BEHAVIOR ANALYSIS - SUMMARY")
print("="*50)

print("\n✅ What we learned:")
print("   1. Speeding detection (speed > limit + 10%)")
print("   2. Harsh braking detection (decel > 3 m/s²)")
print("   3. Harsh cornering detection (lateral accel > 2 m/s²)")
print("   4. Driver scorecard calculation (0-100)")

print("\n📊 Results:")
print(f"   Speeding Events:    {len(speeding_events)}")
print(f"   Harsh Braking:      {len(braking_events)}")
print(f"   Harsh Cornering:    {len(cornering_events)}")
print(f"   Total Violations:   {len(speeding_events) + len(braking_events) + len(cornering_events)}")
print(f"   Driver Score:       {score}/100")
print(f"   Rating:             {rating}")

print("\n📚 Key Concepts:")
print("   - Threshold-based detection")
print("   - Driver performance metrics")
print("   - Scorecard calculation")
print("   - Behavioral analytics")

print("\n🚀 Next: Phase 5 - Dashboard!")
print("   - Real-time visualization")
print("   - Web interface")
print("   - Driver monitoring")

 Multiple Driver Analysis

In [ ]:
# ============================================================
# 📊 SECTION: MULTIPLE DRIVERS ANALYSIS
# ============================================================
# Generate multiple drivers with different driving styles
# and compare their performance

print("\n" + "="*50)
print("📊 GENERATING MULTIPLE DRIVERS")
print("="*50)

def generate_driver_data(driver_id, driving_style='normal'):
    """
    Generate driving data for a specific driver.
    
    driving_style:
    - 'safe': Very few violations
    - 'normal': Some violations
    - 'aggressive': Many violations
    - 'unsafe': Very dangerous
    """
    np.random.seed(driver_id * 100)  # Different seed per driver
    
    # Number of violations based on style
    if driving_style == 'safe':
        speed_violations = np.random.poisson(0.5)
        braking_violations = np.random.poisson(0.3)
        cornering_violations = np.random.poisson(0.2)
    elif driving_style == 'normal':
        speed_violations = np.random.poisson(3)
        braking_violations = np.random.poisson(2)
        cornering_violations = np.random.poisson(1)
    elif driving_style == 'aggressive':
        speed_violations = np.random.poisson(8)
        braking_violations = np.random.poisson(6)
        cornering_violations = np.random.poisson(4)
    else:  # unsafe
        speed_violations = np.random.poisson(15)
        braking_violations = np.random.poisson(12)
        cornering_violations = np.random.poisson(10)
    
    # Calculate score
    score = calculate_driver_score(speed_violations, braking_violations, cornering_violations)
    
    return {
        'driver_id': driver_id,
        'style': driving_style,
        'speed_violations': speed_violations,
        'braking_violations': braking_violations,
        'cornering_violations': cornering_violations,
        'total_violations': speed_violations + braking_violations + cornering_violations,
        'score': score
    }

# Generate multiple drivers
print("👥 Creating driver profiles...")

drivers_data = []

# Safe drivers
for i in range(3):
    drivers_data.append(generate_driver_data(i+1, 'safe'))

# Normal drivers
for i in range(3):
    drivers_data.append(generate_driver_data(i+4, 'normal'))

# Aggressive drivers
for i in range(2):
    drivers_data.append(generate_driver_data(i+7, 'aggressive'))

# Unsafe driver (our original)
drivers_data.append(generate_driver_data(10, 'unsafe'))

print(f"✅ Created {len(drivers_data)} drivers")

# ============================================================
# 📊 SECTION: DISPLAY ALL DRIVERS
# ============================================================
print("\n📊 DRIVER SCORECARD COMPARISON")
print("="*60)
print(f"{'Driver':<10} {'Style':<12} {'Speed':<6} {'Brake':<6} {'Corner':<6} {'Total':<6} {'Score':<6} {'Rating':<12}")
print("-"*60)

for driver in drivers_data:
    score = driver['score']
    
    if score >= 90:
        rating = "⭐ EXCELLENT"
    elif score >= 70:
        rating = "👍 GOOD"
    elif score >= 50:
        rating = "⚠️ NEEDS IMPROVEMENT"
    else:
        rating = "🚨 UNSAFE"
    
    print(f"{driver['driver_id']:<10} {driver['style']:<12} {driver['speed_violations']:<6} {driver['braking_violations']:<6} {driver['cornering_violations']:<6} {driver['total_violations']:<6} {driver['score']:<6} {rating:<12}")

# ============================================================
# 📊 SECTION: VISUALIZE ALL DRIVERS
# ============================================================
print("\n📊 VISUALIZING ALL DRIVERS")
print("="*50)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Extract data
driver_ids = [d['driver_id'] for d in drivers_data]
scores = [d['score'] for d in drivers_data]
total_violations = [d['total_violations'] for d in drivers_data]

# 1. Score Bar Chart
colors = ['green' if s >= 70 else 'orange' if s >= 50 else 'red' for s in scores]
axes[0, 0].bar(driver_ids, scores, color=colors)
axes[0, 0].axhline(y=90, color='green', linestyle='--', label='Excellent')
axes[0, 0].axhline(y=70, color='orange', linestyle='--', label='Good')
axes[0, 0].axhline(y=50, color='red', linestyle='--', label='Needs Improvement')
axes[0, 0].set_xlabel('Driver ID')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Driver Scores')
axes[0, 0].set_ylim(0, 105)
axes[0, 0].legend()

# 2. Violations Breakdown
drivers_to_show = drivers_data[:9]
x = range(len(drivers_to_show))
speed_vals = [d['speed_violations'] for d in drivers_to_show]
brake_vals = [d['braking_violations'] for d in drivers_to_show]
corner_vals = [d['cornering_violations'] for d in drivers_to_show]

axes[0, 1].bar(x, speed_vals, label='Speeding', color='#ff6b6b')
axes[0, 1].bar(x, brake_vals, bottom=speed_vals, label='Braking', color='#ffd93d')
axes[0, 1].bar(x, corner_vals, bottom=[speed_vals[i] + brake_vals[i] for i in range(len(x))], 
               label='Cornering', color='#6bcb77')
axes[0, 1].set_xlabel('Driver ID')
axes[0, 1].set_ylabel('Violations')
axes[0, 1].set_title('Violations by Driver')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels([d['driver_id'] for d in drivers_to_show])
axes[0, 1].legend()

# 3. Score vs Total Violations
axes[1, 0].scatter(total_violations, scores, s=100, c=colors, alpha=0.7)
for driver in drivers_data:
    axes[1, 0].annotate(f"Driver {driver['driver_id']}", 
                        (driver['total_violations'], driver['score']), 
                        fontsize=8)
axes[1, 0].set_xlabel('Total Violations')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Score vs Violations')
axes[1, 0].grid(True, alpha=0.3)

# 4. Driver Distribution
styles = ['safe', 'normal', 'aggressive', 'unsafe']
style_counts = [0, 0, 0, 0]
for d in drivers_data:
    if d['style'] == 'safe':
        style_counts[0] += 1
    elif d['style'] == 'normal':
        style_counts[1] += 1
    elif d['style'] == 'aggressive':
        style_counts[2] += 1
    else:
        style_counts[3] += 1

axes[1, 1].pie(style_counts, labels=styles, autopct='%1.1f%%', 
               colors=['green', 'blue', 'orange', 'red'], startangle=90)
axes[1, 1].set_title('Driver Style Distribution')

plt.tight_layout()
plt.show()

# ============================================================
# 📊 SECTION: MULTI-DRIVER SUMMARY
# ============================================================
print("\n" + "="*50)
print("📊 MULTI-DRIVER ANALYSIS - SUMMARY")
print("="*50)

# Count drivers by category
excellent = sum(1 for d in drivers_data if d['score'] >= 90)
good = sum(1 for d in drivers_data if 70 <= d['score'] < 90)
needs_improvement = sum(1 for d in drivers_data if 50 <= d['score'] < 70)
unsafe = sum(1 for d in drivers_data if d['score'] < 50)

print(f"\n📊 Driver Distribution:")
print(f"   ⭐ Excellent (90+):   {excellent} drivers")
print(f"   👍 Good (70-89):      {good} drivers")
print(f"   ⚠️ Needs Improvement (50-69): {needs_improvement} drivers")
print(f"   🚨 Unsafe (<50):      {unsafe} drivers")

print(f"\n📊 Overall Stats:")
avg_score = np.mean(scores)
print(f"   Average Score:        {avg_score:.1f}/100")
print(f"   Highest Score:        {max(scores)}/100")
print(f"   Lowest Score:         {min(scores)}/100")
print(f"   Total Drivers:        {len(drivers_data)}")
print(f"   Total Violations:     {sum(total_violations)}")

 Save Data to CSV
 

In [ ]:
# Cell: Save Driver Data to CSV
import pandas as pd
import os

# Create folders if they don't exist
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# Your driver data
data = {
    'Driver ID': [1, 2, 3, 4, 5, 6, 7, 8, 10],
    'Driving Style': ['Safe', 'Safe', 'Safe', 'Normal', 'Normal', 'Normal', 
                      'Aggressive', 'Aggressive', 'Unsafe'],
    'Score': [100, 98, 98, 82, 79, 77, 45, 35, 0],
    'Speeding Violations': [0, 0, 0, 2, 3, 3, 8, 10, 15],
    'Braking Violations': [0, 0, 0, 1, 1, 2, 5, 6, 13],
    'Cornering Violations': [0, 1, 1, 2, 2, 2, 7, 8, 9],
}

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv('data/raw/driver_data_raw.csv', index=False)
df.to_csv('data/processed/driver_data_processed.csv', index=False)

print("✅ Data saved successfully!")
print(f"   Raw: data/raw/driver_data_raw.csv")
print(f"   Processed: data/processed/driver_data_processed.csv")

Dashboard Loading CSV

In [ ]:
import pandas as pd
import os

print("="*50)
print("📊 CHECKING CSV FILE")
print("="*50)

csv_path = 'data/processed/driver_data_processed.csv'

# Check if file exists
if os.path.exists(csv_path):
    print(f"✅ CSV file found at: {os.path.abspath(csv_path)}")
    
    # Read and check data
    df = pd.read_csv(csv_path)
    print(f"✅ Loaded {len(df)} rows")
    print(f"✅ Columns: {list(df.columns)}")
    print("\n📋 Data:")
    print(df)
else:
    print(f"❌ CSV file NOT found at: {os.path.abspath(csv_path)}")
    
    # Check if data folder exists
    print(f"📁 Data folder exists: {os.path.exists('data')}")
    if os.path.exists('data'):
        print(f"📁 Raw folder exists: {os.path.exists('data/raw')}")
        print(f"📁 Processed folder exists: {os.path.exists('data/processed')}")

Display in Dashboard

In [ ]:
# Cell: Create CSV with Real Data
import pandas as pd
import os

# Make sure we're in the right directory
project_dir = r'C:\Users\user\Desktop\CAREER_PORTFOLIO\CAREER_PORTFOLIO\07_GITHUB_PROJECTS\02_Transportation_Security_System'
os.chdir(project_dir)
print(f"📂 Working in: {os.getcwd()}")

# Create folders
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# Your REAL data (with values!)
data = {
    'Driver ID': [1, 2, 3, 4, 5, 6, 7, 8, 10],
    'Driving Style': ['Safe', 'Safe', 'Safe', 'Normal', 'Normal', 'Normal', 
                      'Aggressive', 'Aggressive', 'Unsafe'],
    'Score': [100, 98, 98, 82, 79, 77, 45, 35, 0],
    'Speeding Violations': [0, 0, 0, 2, 3, 3, 8, 10, 15],
    'Braking Violations': [0, 0, 0, 1, 1, 2, 5, 6, 13],
    'Cornering Violations': [0, 1, 1, 2, 2, 2, 7, 8, 9],
}

df = pd.DataFrame(data)

# Add Status column
def get_status(score):
    if score >= 90:
        return "✅ Excellent"
    elif score >= 70:
        return "👍 Good"
    elif score >= 50:
        return "⚠️ Needs Improvement"
    else:
        return "🚨 Unsafe"

df['Status'] = df['Score'].apply(get_status)
df['Total Violations'] = df['Speeding Violations'] + df['Braking Violations'] + df['Cornering Violations']

# Save to CSV
df.to_csv('data/processed/driver_data_processed.csv', index=False)
df.to_csv('data/raw/driver_data_raw.csv', index=False)

print("✅ Data saved successfully!")
print("\n📋 Data:")
print(df.to_string(index=False))

print("\n📁 File locations:")
print(f"   Processed: {os.path.abspath('data/processed/driver_data_processed.csv')}")
print(f"   Raw: {os.path.abspath('data/raw/driver_data_raw.csv')}")

In [ ]:
import pandas as pd
import os

print("="*50)
print("📊 CHECKING CSV FILE")
print("="*50)

csv_path = 'data/processed/driver_data_processed.csv'

# Check if file exists
if os.path.exists(csv_path):
    print(f"✅ CSV file found at: {os.path.abspath(csv_path)}")
    
    # Read and check data
    df = pd.read_csv(csv_path)
    print(f"✅ Loaded {len(df)} rows")
    print(f"✅ Columns: {list(df.columns)}")
    print("\n📋 Data:")
    print(df)
else:
    print(f"❌ CSV file NOT found at: {os.path.abspath(csv_path)}")
    
    # Check if data folder exists
    print(f"📁 Data folder exists: {os.path.exists('data')}")
    if os.path.exists('data'):
        print(f"📁 Raw folder exists: {os.path.exists('data/raw')}")
        print(f"📁 Processed folder exists: {os.path.exists('data/processed')}")